In [2]:
import torch

In [6]:
import torch.nn as nn

class WithSuperInit(nn.Module):
    def __init__(self):
        super(WithSuperInit, self).__init__()
        # Not calling parent's init
        self.layer = nn.Linear(10, 10)

    def forward(self, x):
        return self.layer(x)

model = WithoutSuperInit()
print(list(model.parameters()))


[Parameter containing:
tensor([[ 0.1053,  0.2896, -0.2123,  0.2350, -0.2475, -0.1589,  0.1906, -0.1123,
          0.2587, -0.0879],
        [-0.1395, -0.0585,  0.1328, -0.1900, -0.1898,  0.0748, -0.2170,  0.1115,
         -0.0150,  0.1072],
        [ 0.1013, -0.2092,  0.1524,  0.2371, -0.1605,  0.1393, -0.0511, -0.1700,
          0.2701, -0.2690],
        [ 0.1655,  0.0859,  0.0022, -0.1655,  0.1419, -0.1899,  0.1392, -0.0435,
         -0.0072,  0.2367],
        [ 0.2745,  0.2837, -0.3024, -0.2332,  0.1013, -0.0337, -0.1597,  0.2880,
         -0.0890,  0.2074],
        [-0.2101,  0.2423,  0.2457,  0.0320, -0.0238, -0.2121,  0.0071,  0.2052,
          0.2071,  0.0338],
        [-0.0468, -0.2178,  0.2403,  0.3100,  0.0619, -0.2890,  0.2473,  0.1013,
          0.1393, -0.1105],
        [ 0.0704,  0.0412,  0.0221,  0.1040, -0.1309, -0.1005,  0.0180,  0.2650,
          0.2822,  0.0662],
        [ 0.2259,  0.0331, -0.0300,  0.0580,  0.0233, -0.2940, -0.2349, -0.1846,
         -0.2426,  0.291

In [7]:
import torch
import torch.nn as nn

class DeepSet(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(DeepSet, self).__init__()
        # Transformation network
        self.phi = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU()
        )
        
        # Aggregation and further processing
        self.rho = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )
    
    def forward(self, x):
        # x is of shape [batch_size, set_size, input_dim]
        
        # Transform each element in the set
        x_transformed = self.phi(x)
        
        # Sum over the set dimension (aggregation)
        x_aggregated = x_transformed.sum(dim=1)
        
        # Further processing
        out = self.rho(x_aggregated)
        
        return out

# Create the model
model = DeepSet(input_dim=4, hidden_dim=64, output_dim=1)

# Example input: 10 sets each with 5 particles described by 4 features
x_example = torch.rand((10, 5, 4))

# Get the model output
output = model(x_example)
print(output)


tensor([[-0.0405],
        [-0.1051],
        [-0.0806],
        [-0.0430],
        [-0.0891],
        [-0.0847],
        [-0.0575],
        [-0.0336],
        [-0.0386],
        [-0.0671]], grad_fn=<AddmmBackward0>)


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

# Define the neural network model for jet discrimination
class JetClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super(JetClassifier, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 2),  # Two classes: quark or gluon
            nn.Softmax(dim=1)
        )
    
    def forward(self, x):
        return self.network(x)

# Generate synthetic data
num_samples = 1000

# Quark jets
quark_multiplicity = torch.randint(5, 15, (num_samples, 1))
quark_pT = torch.randn(num_samples, 1) + 3  # Centered around 3
quark_radius = torch.rand(num_samples, 1) * 0.3  # Typically smaller

# Gluon jets
gluon_multiplicity = torch.randint(15, 30, (num_samples, 1))
gluon_pT = torch.randn(num_samples, 1) + 2  # Broader distribution
gluon_radius = torch.rand(num_samples, 1) * 0.5 + 0.3  # Typically larger

# Combine data and create labels
data = torch.cat([torch.cat([quark_multiplicity, quark_pT, quark_radius], dim=1),
                  torch.cat([gluon_multiplicity, gluon_pT, gluon_radius], dim=1)])
labels = torch.cat([torch.zeros(num_samples), torch.ones(num_samples)]).long()

# Train the classifier
model = JetClassifier(input_dim=3, hidden_dim=64)
optimizer = optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(100):
    optimizer.zero_grad()
    outputs = model(data)
    loss = loss_fn(outputs, labels)
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        _, predicted = torch.max(outputs, 1)
        accuracy = (predicted == labels).float().mean()
        print(f"Epoch {epoch+1}, Loss: {loss.item()}, Accuracy: {accuracy.item() * 100:.2f}%")

# Test
test_jet = torch.tensor([[10, 3.5, 0.25]])  # A jet with multiplicity 10, pT 3.5, and radius 0.25
probabilities = model(test_jet)
print(f"Probability of being Quark-initiated: {probabilities[0][0].item() * 100:.2f}%")
print(f"Probability of being Gluon-initiated: {probabilities[0][1].item() * 100:.2f}%")


Epoch 10, Loss: 0.4326976537704468, Accuracy: 93.20%
Epoch 20, Loss: 0.3825379014015198, Accuracy: 93.20%
Epoch 30, Loss: 0.3732520639896393, Accuracy: 93.95%
Epoch 40, Loss: 0.36382123827934265, Accuracy: 94.90%
Epoch 50, Loss: 0.3545997142791748, Accuracy: 95.90%
Epoch 60, Loss: 0.3440043032169342, Accuracy: 96.90%
Epoch 70, Loss: 0.3316679000854492, Accuracy: 98.40%
Epoch 80, Loss: 0.3224292993545532, Accuracy: 99.75%
Epoch 90, Loss: 0.3195016384124756, Accuracy: 99.85%
Epoch 100, Loss: 0.3174912929534912, Accuracy: 99.95%
Probability of being Quark-initiated: 99.95%
Probability of being Gluon-initiated: 0.05%
